In [1]:
import random

from spiral import Spiral

sp = Spiral(overrides={
    "keys_cache.enabled": "1",
    "keys_cache.memory_capacity_bytes": "1073741824",  # 1 GiB
    "keys_cache.disk_capacity_bytes": "0",
    "fragments_cache.enabled": "1",
    "fragments_cache.memory_capacity_bytes": "4294967296",  # 4 GiB
    "fragments_cache.disk_capacity_bytes": "0",
    "manifests_cache.enabled": "1",
    "manifests_cache.memory_capacity_bytes": "0",
    "manifests_cache.disk_capacity_bytes": "1073741824",  # 1GiB
})

In [2]:
project = sp.project("enigma-spiral-poc-2-724186")

In [17]:
project.list_tables()

[TableResource(id='table_35vkxf', project_id='enigma-spiral-poc-2-724186', dataset='default', table='stim_constants'),
 TableResource(id='table_3ifkzf', project_id='enigma-spiral-poc-2-724186', dataset='default', table='session_reconstructed_video_metadata'),
 TableResource(id='table_3lzdom', project_id='enigma-spiral-poc-2-724186', dataset='default', table='spike_data'),
 TableResource(id='table_4cpkjo', project_id='enigma-spiral-poc-2-724186', dataset='default', table='vidtok_embeddings'),
 TableResource(id='table_69tszf', project_id='enigma-spiral-poc-2-724186', dataset='default', table='lfp_probe_metadata'),
 TableResource(id='table_78ttpe', project_id='enigma-spiral-poc-2-724186', dataset='default', table='spike_data_by_unit'),
 TableResource(id='table_ekccal', project_id='enigma-spiral-poc-2-724186', dataset='default', table='stim_trial_info'),
 TableResource(id='table_jxze31', project_id='enigma-spiral-poc-2-724186', dataset='default', table='lfp_data'),
 TableResource(id='table

In [4]:
tbl_session_reconstructed_video_metadata = project.table("session_reconstructed_video_metadata")
tbl_spike_data_by_time = project.table("spike_data_by_time")
tbl_vidtok_embeddings = project.table("vidtok_embeddings")
tbl_vjepa_embeddings = project.table("vjepa_embeddings")
tbl_behavior_adc = project.table("behavior_adc")
tbl_stim_events = project.table("stim_events")

In [5]:
# Random explorations.
tbl_behavior_adc.schema()

Schema({session_id=utf8?, timestamp=i64?, eye_x_px_offset_center=f64?, eye_y_px_offset_center=f64?, neuropixel_sync_in=f64?, photodiode=f64?, pupil_size_in=f64?, reward_input=f64?})

In [6]:
# Random explorations.
tbl_session_reconstructed_video_metadata.to_polars_lazy_frame().head().collect()

session_id,reconstruction_frame_number,bg_color,display,displayed_movie,displayed_movie_frame_number,source_movie,source_movie_frame_number,timestamp,trial_idx,fixation_dot
str,i64,list[f64],str,str,f64,str,f64,i64,i64,struct[4]
"""Goliath_2025-10-20_20-40-05""",0,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52431216,0,"{[255, 0, 0],[0, 0],0,25}"
"""Goliath_2025-10-20_20-40-05""",1,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52439557,0,"{[255, 0, 0],[0, 0],0,25}"
"""Goliath_2025-10-20_20-40-05""",2,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52447899,0,"{[255, 0, 0],[0, 0],0,25}"
"""Goliath_2025-10-20_20-40-05""",3,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52456241,0,"{[255, 0, 0],[0, 0],0,25}"
"""Goliath_2025-10-20_20-40-05""",4,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52464582,0,"{[255, 0, 0],[0, 0],0,25}"


In [7]:
import pyarrow as pa
from spiral import Shard
from spiral.core.table import KeyRange  # TODO(marko): Fix this.
from spiral.core.table.spec import Key  # TODO(marko): Fix this.

rd = random.seed(43)

def ranges_to_shards(tbl, list_ranges) -> list[Shard]:
    shards = []
    for r in list_ranges:
        st = tbl.key(r["session_id"], r["start"])
        ed = Key(bytes(tbl.key(r["session_id"], r["end"])) + bytes(Key.max()))
        key_range = KeyRange(begin=st, end=ed)
        shards.append(Shard(key_range, None))
    return shards

time_ranges = sp.scan({
    "session_id": tbl_session_reconstructed_video_metadata["session_id"],
    "start": tbl_session_reconstructed_video_metadata["timestamp"],
    "end": tbl_session_reconstructed_video_metadata["timestamp"] + 1_000_000,
}).to_table().to_pylist()
sessions_shards = ranges_to_shards(tbl_session_reconstructed_video_metadata, time_ranges)

sessions_shards[0]

Shard { key_range: KeyRange { begin: Key(\x02Goliath\x5f2025\x2d10\x2d20\x5f20\x2d40\x2d05\x00\x18\x03\x20\x09p), end: Key(\x02Goliath\x5f2025\x2d10\x2d20\x5f20\x2d40\x2d05\x00\x18\x03\x2fK\xb0⊤) }, cardinality: None }

In [8]:
from spiral import Sampler

def random_boolean_mask(arr, n_samples):
    """
    Create a boolean mask with n_samples random True values.

    Parameters:
    -----------
    arr : pyarrow.Array
        Input Arrow array
    n_samples : int
        Number of random samples to select

    Returns:
    --------
    pyarrow.Array
        Boolean mask array
    """
    length = len(arr)

    # Ensure n_samples doesn't exceed array length
    n_samples = min(n_samples, length)

    # Create boolean mask with all False
    mask = np.zeros(length, dtype=bool)

    # Randomly select indices
    random_indices = np.random.choice(length, size=n_samples, replace=False)

    # Set selected indices to True
    mask[random_indices] = True

    # Convert to Arrow array
    return pa.array(mask)

def behavior_sampler_function(array: pa.Array) -> pa.Array:
    return pa.array([i % 10 == 0 for i in range(len(array))])

def modality_sampler_function(array: pa.Array) -> pa.Array:
    # NOTE: can use modalities array.field("modality")
    # Only key columns are available, let us know if you need to pull in other columns.
    return random_boolean_mask(array, 8)

behavior_sampler = Sampler(behavior_sampler_function)
modality_sampler = Sampler(modality_sampler_function)

In [9]:
vidtok_embeddings_scan = sp.scan(tbl_vidtok_embeddings["tensor"], where=tbl_vidtok_embeddings["modality"] == "rgb")
vjepa_embeddings_scan = sp.scan(tbl_vjepa_embeddings["tensor"], where=tbl_vjepa_embeddings["layer"] == 4)
behavior_scan = sp.scan(tbl_behavior_adc[["pupil_size_in", "eye_x_px_offset_center", "eye_y_px_offset_center"]])

In [10]:
import numpy as np

def stack_item(item):
    """
    Convert a single batch item to numpy arrays (most efficient version).
    Handles variable lengths: behavior can be 600 or 601, vidtok can have 7 or 8 embeddings.

    Args:
        item: Tuple of (behavior_batch, embeddings_batch)

    Returns:
        dict[str, np.ndarray]: Dictionary with 'behavior' and 'vidtok' arrays
    """
    behavior_batch = item[0]
    embeddings_batch = item[1]

    # Stack behavior data (600 or 601, 3)
    behavior_array = np.column_stack([
        behavior_batch["pupil_size_in"].to_numpy(zero_copy_only=False),
        behavior_batch["eye_x_px_offset_center"].to_numpy(zero_copy_only=False),
        behavior_batch["eye_y_px_offset_center"].to_numpy(zero_copy_only=False)
    ])

    # Stack vidtok embeddings
    tensor_column = embeddings_batch["tensor"]
    num_vidtoks = len(tensor_column)

    # Double flatten: list<list<float>> -> flat float array
    flattened_once = pa.ListArray.flatten(tensor_column)
    flattened_twice = pa.ListArray.flatten(flattened_once)

    # Zero-copy to numpy
    vidtok_flat = flattened_twice.to_numpy(zero_copy_only=False)

    # Infer number of rows per vidtok (should be 1590)
    num_vidtok_rows = len(vidtok_flat) // (num_vidtoks * 16)

    # Reshape and transpose: (num_vidtoks, num_vidtok_rows, 16) -> (num_vidtok_rows, 16, num_vidtoks)
    vidtok_array = vidtok_flat.reshape(num_vidtoks, num_vidtok_rows, 16).transpose(1, 2, 0)

    # Stack vjepa embeddings (15 rows of (33792, 24) -> (1408, 24, 24, 15))
    vjepa_tensor_column = vjepa_batch["tensor"]
    num_vjepa = len(vjepa_tensor_column)  # Should be 15

    # Double flatten: list<list<float>> -> flat float array
    vjepa_flattened_once = pa.ListArray.flatten(vjepa_tensor_column)
    vjepa_flattened_twice = pa.ListArray.flatten(vjepa_flattened_once)

    # Zero-copy to numpy
    vjepa_flat = vjepa_flattened_twice.to_numpy(zero_copy_only=False)

    # Reshape: (15 * 33792 * 24) -> (15, 33792, 24) -> (15, 1408, 24, 24) -> (1408, 24, 24, 15)
    vjepa_array = vjepa_flat.reshape(num_vjepa, 33792, 24).reshape(num_vjepa, 1408, 24, 24).transpose(1, 2, 3, 0)

    return {
        "behavior": behavior_array,  # Shape: (600 or 601, 3)
        "vidtok": vidtok_array,      # Shape: (1590, 16, 7 or 8)
        "vjepa": vjepa_array         # Shape: (1408, 24, 24, 15)
    }

In [16]:
import tqdm

batch_readahead = 2

behavior_loader = behavior_scan.to_record_batches(shards=sessions_shards, sampler=behavior_sampler, batch_readahead=batch_readahead)
# TODO(marko): Run with modality_sampler when sampler pushdown bug is fixed.
vidtok_embeddings_loader = vidtok_embeddings_scan.to_record_batches(shards=sessions_shards, batch_readahead=batch_readahead)
vjepa_embeddings_loader = vjepa_embeddings_scan.to_record_batches(shards=sessions_shards, batch_readahead=batch_readahead)

for item in tqdm.tqdm(zip(behavior_loader, vidtok_embeddings_loader, vjepa_embeddings_loader)):
    sample = stack_item(item)

    # print({
    #     key: value.shape for key, value in sample.items()
    # })
    # break

KeyboardInterrupt: 